**© Copyright AIDENTIFY. All rights reserved.**

본 자료는 **멀티캠퍼스 LLM 파인튜닝 과정** 수강생을 위해 제작되었으며, 강의 목적으로만 사용 가능합니다.  
무단 복제, 배포, 상업적 이용을 금지합니다.

---

# 🎯 생성 AI와 LLM 개요

## Part 1 | Sessions 1-2: 생성 AI / 트랜스포머 구조 / 토큰화 실습 / 생성 전략

---

### 📋 학습 목표

- 🔹 생성 AI(Generative AI)의 개념과 LLM의 위치를 이해합니다
- 🔹 트랜스포머(Transformer) 아키텍처의 핵심 구조를 파악합니다
- 🔹 토큰화(Tokenization)의 원리와 다양한 방식을 실습합니다
- 🔹 텍스트 생성 전략(Greedy, Beam Search, Sampling 등)을 이해합니다
- 🔹 한국어와 영어의 토큰 효율 차이를 비교합니다

### 📦 필요 라이브러리

```
tiktoken, transformers, torch
```

### ⏱️ 예상 소요 시간: 약 90분

---

In [ ]:
# 필요 라이브러리 설치 (이미 설치되어 있다면 스킵)
# !pip install tiktoken transformers torch

import tiktoken
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

print("✅ 라이브러리 임포트 완료!")
print(f"  - tiktoken 버전: {tiktoken.__version__}")
print(f"  - torch 버전: {torch.__version__}")
print(f"  - CUDA 사용 가능: {torch.cuda.is_available()}")

---

## 🎯 생성 AI(Generative AI)란?

생성 AI는 **새로운 콘텐츠(텍스트, 이미지, 코드 등)를 생성**할 수 있는 인공지능 모델입니다.

### 생성 AI의 주요 분류

| 분류 | 설명 | 대표 모델 |
|------|------|----------|
| 텍스트 생성 | 자연어 텍스트 생성 | GPT-4, Claude, Gemini |
| 이미지 생성 | 텍스트→이미지 생성 | DALL-E 3, Midjourney, Stable Diffusion |
| 코드 생성 | 프로그래밍 코드 생성 | Codex, GitHub Copilot |
| 음성/음악 | 오디오 콘텐츠 생성 | MusicGen, Bark |
| 동영상 | 비디오 콘텐츠 생성 | Sora, Runway |

### LLM(Large Language Model)의 위치

```
인공지능 (AI)
  └── 머신러닝 (ML)
      └── 딥러닝 (DL)
          └── 생성 AI (Generative AI)
              └── LLM (Large Language Model)
                  └── sLLM (Small LLM) ← 우리가 파인튜닝할 모델!
```

### LLM의 핵심 작동 원리

LLM은 본질적으로 **"다음 토큰 예측(Next Token Prediction)"** 을 수행합니다.

```
입력: "오늘 날씨가"
  → 모델이 다음 토큰 확률 계산
  → "좋습니다" (확률 0.35), "맑습니다" (확률 0.25), ...
  → 선택된 토큰을 추가
  → "오늘 날씨가 좋습니다"
```

---

---

## 1️⃣ 트랜스포머(Transformer) 구조 이해

2017년 Google의 "Attention Is All You Need" 논문에서 제안된 트랜스포머는 현대 LLM의 기반입니다.

### 트랜스포머의 핵심 구성요소

```
┌─────────────────────────────────────────────┐
│              Transformer 구조                 │
├─────────────────────────────────────────────┤
│                                             │
│  ┌─────────────┐     ┌─────────────┐       │
│  │  Encoder     │     │  Decoder     │      │
│  │  (인코더)     │     │  (디코더)     │      │
│  │             │     │             │       │
│  │ Self-       │     │ Masked      │       │
│  │ Attention   │────▶│ Self-       │       │
│  │             │     │ Attention   │       │
│  │ Feed        │     │             │       │
│  │ Forward     │     │ Cross-      │       │
│  │             │     │ Attention   │       │
│  │             │     │             │       │
│  │             │     │ Feed        │       │
│  │             │     │ Forward     │       │
│  └─────────────┘     └─────────────┘       │
│                                             │
│  입력 임베딩 + 위치 인코딩 (Positional Encoding) │
└─────────────────────────────────────────────┘
```

### 모델 유형별 분류

| 유형 | 구조 | 대표 모델 | 주요 용도 |
|------|------|----------|----------|
| Encoder-only | 인코더만 사용 | BERT, RoBERTa | 분류, NER, 유사도 |
| Decoder-only | 디코더만 사용 | GPT, LLaMA, Qwen | 텍스트 생성 |
| Encoder-Decoder | 둘 다 사용 | T5, BART | 번역, 요약 |

### Self-Attention (자기 주의 메커니즘)

Self-Attention은 입력 시퀀스의 각 토큰이 **다른 모든 토큰과의 관계**를 계산합니다.

```
Attention(Q, K, V) = softmax(QK^T / √d_k) × V

Q (Query)  : 현재 토큰이 "무엇을 찾고 있는지"
K (Key)    : 각 토큰이 "어떤 정보를 제공하는지"
V (Value)  : 각 토큰의 "실제 정보"
```

---

In [ ]:
# Self-Attention 직관적 이해를 위한 간단한 예시
import torch
import torch.nn.functional as F

print("=" * 60)
print("🔍 Self-Attention 간단 시뮬레이션")
print("=" * 60)

# 3개 토큰, 각 4차원 임베딩으로 가정
tokens = ["나는", "학생", "입니다"]
# 임의의 임베딩 벡터 (실제로는 학습된 값)
embeddings = torch.randn(3, 4)

print(f"\n📌 입력 토큰: {tokens}")
print(f"📌 임베딩 shape: {embeddings.shape} (3개 토큰 × 4차원)")

# Q, K, V 생성 (간단히 같은 임베딩 사용)
Q = embeddings
K = embeddings
V = embeddings

# Attention Score 계산
d_k = Q.shape[-1]
scores = torch.matmul(Q, K.T) / (d_k ** 0.5)
attention_weights = F.softmax(scores, dim=-1)

print(f"\n📊 Attention Weights (각 토큰이 다른 토큰에 주는 관심도):")
for i, token in enumerate(tokens):
    weights_str = ", ".join([f"{tokens[j]}: {attention_weights[i][j]:.3f}" for j in range(len(tokens))])
    print(f"  '{token}' → [{weights_str}]")

# 가중 합 계산
output = torch.matmul(attention_weights, V)
print(f"\n📌 출력 shape: {output.shape}")
print("✅ 각 토큰이 다른 토큰의 정보를 가중 합산하여 새로운 표현을 만들었습니다!")

---

## 2️⃣ 토큰화(Tokenization) 실습 - tiktoken으로 GPT 토큰화 체험

### 토큰화란?

토큰화는 텍스트를 모델이 이해할 수 있는 **숫자 단위(토큰)**로 변환하는 과정입니다.

```
"Hello World" → [15339, 2159] → 모델 입력
"안녕하세요"    → [31585, 233, 42468, 98, 46695, 244] → 모델 입력
```

### 왜 토큰화가 중요한가?

- 🔸 모델의 **입력 길이 제한** (예: 4096 토큰)에 직접 영향
- 🔸 **API 비용**이 토큰 수 기준으로 부과
- 🔸 한국어는 영어보다 더 많은 토큰을 사용 → **비용/효율 차이**

---

In [ ]:
# tiktoken으로 GPT 모델의 토큰화 체험
import tiktoken

print("=" * 60)
print("🔤 tiktoken을 이용한 GPT 토큰화 실습")
print("=" * 60)

# GPT-4에서 사용하는 인코딩
enc = tiktoken.encoding_for_model("gpt-4")
print(f"\n📌 사용 인코딩: {enc.name}")

# 다양한 텍스트로 토큰화 실험
texts = [
    "Hello, World!",
    "안녕하세요, 세계!",
    "I love artificial intelligence.",
    "나는 인공지능을 좋아합니다.",
    "Fine-tuning is the process of adapting a pre-trained model.",
    "파인튜닝은 사전 학습된 모델을 적응시키는 과정입니다.",
]

print("\n" + "-" * 60)
for text in texts:
    tokens = enc.encode(text)
    print(f"\n📝 원문: \"{text}\"")
    print(f"   토큰 ID: {tokens}")
    print(f"   토큰 수: {len(tokens)}")
    
    # 각 토큰을 디코딩하여 확인
    decoded_tokens = [enc.decode([t]) for t in tokens]
    print(f"   토큰 분해: {decoded_tokens}")

print("\n" + "=" * 60)
print("✅ 한국어가 영어보다 더 많은 토큰을 사용하는 것을 확인할 수 있습니다!")

In [ ]:
# 토큰 수 계산 유틸리티 함수
def count_tokens(text, model="gpt-4"):
    """텍스트의 토큰 수를 계산합니다."""
    enc = tiktoken.encoding_for_model(model)
    return len(enc.encode(text))

# 실습: 긴 텍스트의 토큰 수 확인
long_text_en = """Large Language Models (LLMs) are a type of artificial intelligence 
that can understand and generate human-like text. They are trained on vast amounts 
of text data and can perform a wide range of natural language processing tasks."""

long_text_ko = """대규모 언어 모델(LLM)은 인간과 유사한 텍스트를 이해하고 생성할 수 있는 
인공지능의 한 유형입니다. 방대한 양의 텍스트 데이터로 학습되며 다양한 자연어 처리 작업을 
수행할 수 있습니다."""

en_tokens = count_tokens(long_text_en)
ko_tokens = count_tokens(long_text_ko)

print("📊 긴 텍스트 토큰 비교")
print(f"\n🇺🇸 영어: {len(long_text_en)}자 → {en_tokens} 토큰")
print(f"🇰🇷 한국어: {len(long_text_ko)}자 → {ko_tokens} 토큰")
print(f"\n📌 비율: 한국어는 영어보다 약 {ko_tokens/en_tokens:.1f}배 더 많은 토큰 사용")
print("💡 즉, 같은 의미의 텍스트도 한국어가 API 비용이 더 높습니다!")

---

## 3️⃣ Transformers 토크나이저 비교 (BPE, WordPiece)

### 주요 토큰화 알고리즘

| 알고리즘 | 설명 | 사용 모델 |
|---------|------|----------|
| **BPE** (Byte Pair Encoding) | 가장 빈번한 바이트 쌍을 반복 병합 | GPT, LLaMA, Qwen |
| **WordPiece** | BPE와 유사, 가능도(likelihood) 기반 병합 | BERT, DistilBERT |
| **SentencePiece** | 언어 독립적, Unigram/BPE 지원 | T5, ALBERT, XLNet |

### BPE 작동 원리 (간략)

```
1. 초기: 모든 문자를 개별 토큰으로
   "lower" → ["l", "o", "w", "e", "r"]

2. 가장 빈번한 쌍 찾기 & 병합 반복
   ("l", "o") → "lo"  →  ["lo", "w", "e", "r"]
   ("lo", "w") → "low" →  ["low", "e", "r"]
   ("e", "r") → "er"  →  ["low", "er"]

3. 원하는 어휘 크기까지 반복
```

---

In [ ]:
# HuggingFace Transformers 토크나이저 비교
from transformers import AutoTokenizer

print("=" * 60)
print("🔍 다양한 토크나이저 비교 실습")
print("=" * 60)

# 비교할 토크나이저들
tokenizer_names = {
    "GPT-2 (BPE)": "gpt2",
    "BERT (WordPiece)": "bert-base-uncased",
    "BERT 다국어 (WordPiece)": "bert-base-multilingual-cased",
}

test_texts = [
    "Hello, how are you?",
    "안녕하세요, 반갑습니다.",
    "Transformers are amazing for NLP!",
    "트랜스포머는 자연어 처리에 놀라운 성능을 보여줍니다.",
]

for name, model_name in tokenizer_names.items():
    print(f"\n{'=' * 60}")
    print(f"📦 토크나이저: {name} ({model_name})")
    print(f"{'=' * 60}")
    
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    print(f"   어휘 크기: {tokenizer.vocab_size:,}")
    
    for text in test_texts:
        tokens = tokenizer.tokenize(text)
        ids = tokenizer.encode(text, add_special_tokens=False)
        print(f"\n   📝 \"{text}\"")
        print(f"      토큰: {tokens}")
        print(f"      토큰 수: {len(tokens)}")

print("\n" + "=" * 60)
print("✅ 토크나이저마다 같은 텍스트를 다르게 분할하는 것을 확인하세요!")
print("💡 한국어 처리에는 다국어 토크나이저가 더 효율적입니다.")

In [ ]:
# Special Tokens 이해하기
print("=" * 60)
print("🏷️ Special Tokens (특수 토큰) 이해하기")
print("=" * 60)

# BERT 토크나이저의 특수 토큰
bert_tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
text = "Hello world"
encoded = bert_tokenizer.encode(text)  # 특수 토큰 포함
tokens_with_special = bert_tokenizer.convert_ids_to_tokens(encoded)

print(f"\n📌 BERT 특수 토큰:")
print(f"   [CLS] ID: {bert_tokenizer.cls_token_id} → 문장 시작")
print(f"   [SEP] ID: {bert_tokenizer.sep_token_id} → 문장 구분/끝")
print(f"   [PAD] ID: {bert_tokenizer.pad_token_id} → 패딩")
print(f"   [UNK] ID: {bert_tokenizer.unk_token_id} → 미등록 토큰")
print(f"   [MASK] ID: {bert_tokenizer.mask_token_id} → 마스킹")
print(f"\n   입력: \"{text}\"")
print(f"   인코딩 (특수 토큰 포함): {tokens_with_special}")
print(f"   토큰 ID: {encoded}")

# GPT-2 토크나이저의 특수 토큰
gpt2_tokenizer = AutoTokenizer.from_pretrained("gpt2")
print(f"\n📌 GPT-2 특수 토큰:")
print(f"   EOS (End of Sequence): '{gpt2_tokenizer.eos_token}' (ID: {gpt2_tokenizer.eos_token_id})")
print(f"   BOS (Begin of Sequence): '{gpt2_tokenizer.bos_token}' (ID: {gpt2_tokenizer.bos_token_id})")

print("\n✅ 특수 토큰은 모델이 문장의 구조를 이해하는 데 중요한 역할을 합니다!")

---

## 4️⃣ 텍스트 생성 전략 (Decoding Strategies)

LLM은 다음 토큰의 **확률 분포**를 출력합니다. 이 확률에서 어떻게 토큰을 선택하느냐에 따라 생성 결과가 달라집니다.

### 토큰 생성 과정

```
입력: "나는"
   ↓ 모델이 다음 토큰의 확률 분포 출력
   학생(0.35)  선생(0.20)  사람(0.15)  매우(0.10)  ...
   ↓ 여기서 어떤 토큰을 고를 것인가? = 생성 전략!
```

### 주요 생성 전략 비교

| 전략 | 설명 | 특징 | 용도 | ChatGPT/Claude |
|------|------|------|------|----------------|
| **Greedy** | 매번 확률 1위만 선택 | 결정적, 반복 발생 가능 | 번역, 코드 생성 | ✗ |
| **Beam Search** | 상위 k개 경로를 끝까지 동시 탐색 | 전체 확률 최적화, 느림 | 번역, 음성인식 | ✗ |
| **Top-k Sampling** | 상위 k개 토큰 중 랜덤 선택 | k 고정이라 유연성 부족 | 창작 | △ |
| **Top-p (Nucleus)** | 누적 확률 p%까지의 토큰 중 랜덤 선택 | 동적 후보 수, 가장 많이 사용 | 범용 | ✅ 주로 사용 |
| **Temperature** | 확률 분포의 날카로움 조절 | 다른 전략과 함께 사용 | 전략이 아닌 파라미터 | ✅ 함께 사용 |

### 각 전략의 동작 예시

다음 토큰 확률: 학생(0.35) 선생(0.20) 사람(0.15) 매우(0.10) 정말(0.08) ...

```
Greedy:       "학생" 선택 (1위니까, 항상 같은 결과)

Beam Search:  "학생", "선생", "사람" 3개 경로를 끝까지 추적
              → 최종 문장 확률이 가장 높은 경로 선택

Top-k (k=3):  상위 3개 [학생, 선생, 사람] 중 랜덤 선택
              → 확률 분포: 학생(50%) 선생(29%) 사람(21%)

Top-p (p=0.7): 누적 확률 70%까지 포함
              → 학생(0.35) + 선생(0.20) + 사람(0.15) = 0.70 → 이 3개 중 랜덤
              → 확률 분포에 따라 후보 수가 동적으로 변함!
```

### Top-k vs Top-p 핵심 차이

```
상황 1: 확률이 한 토큰에 집중 ("1+1=")
  "2"(0.95)  "3"(0.02)  "4"(0.01) ...
  Top-k(k=10): 불필요한 9개도 후보에 포함 → 오답 가능!
  Top-p(p=0.95): "2" 하나만 선택 → 정확!

상황 2: 확률이 고르게 분산 ("오늘 저녁은")
  "치킨"(0.12) "피자"(0.11) "파스타"(0.10) "국밥"(0.09) ...
  Top-k(k=3): 3개만 후보 → 다양성 부족
  Top-p(p=0.7): 7~8개까지 후보 → 다양한 선택 가능!
```

→ **Top-p가 상황에 따라 후보 수를 자동 조절**하기 때문에 ChatGPT/Claude가 이 방식을 사용합니다.

### Temperature의 효과

Temperature는 전략이 아니라 **확률 분포를 조절하는 파라미터**입니다.

```
원래 확률:         학생(0.35)  선생(0.20)  사람(0.15)  기타(0.30)

Temperature=0.1:  학생(0.92)  선생(0.05)  사람(0.02)  기타(0.01)
                  → 1위가 거의 확정 (Greedy와 비슷)

Temperature=1.0:  학생(0.35)  선생(0.20)  사람(0.15)  기타(0.30)
                  → 원래 분포 그대로

Temperature=2.0:  학생(0.28)  선생(0.24)  사람(0.22)  기타(0.26)
                  → 거의 균등 → 랜덤에 가까움
```

| Temperature | 효과 | 비유 |
|------------|------|------|
| 0.1 | 거의 확정적 | 교과서 답안 |
| 0.7 | 적당한 다양성 (기본 권장값) | 자연스러운 대화 |
| 1.0 | 원래 확률 분포 그대로 | 자유로운 창작 |
| 1.5+ | 매우 랜덤, 비문 가능 | 횡설수설 |

---


In [ ]:
# GPT-2 모델로 생성 전략 비교 실습
# (GPT-2는 작은 모델이라 CPU에서도 실행 가능)
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

print("=" * 60)
print("🎲 텍스트 생성 전략 비교 실습 (GPT-2)")
print("=" * 60)
print("\n⏳ GPT-2 모델 로딩 중... (최초 실행 시 다운로드 필요)")

model_name = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)
model.eval()

# pad_token 설정
tokenizer.pad_token = tokenizer.eos_token

print("✅ 모델 로딩 완료!")
print(f"   모델 파라미터 수: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# 다양한 생성 전략으로 텍스트 생성
prompt = "The future of artificial intelligence is"
input_ids = tokenizer.encode(prompt, return_tensors="pt")
attention_mask = torch.ones_like(input_ids)

print(f"📝 프롬프트: \"{prompt}\"\n")
print("=" * 60)

# 공통 인자 (경고 방지)
gen_kwargs = dict(attention_mask=attention_mask, pad_token_id=tokenizer.eos_token_id)

# 1. Greedy Decoding
print("\n1️⃣ Greedy Decoding (항상 가장 높은 확률 선택):")
with torch.no_grad():
    output = model.generate(
        input_ids, 
        max_new_tokens=50, 
        do_sample=False,  # Greedy
        **gen_kwargs,
    )
print(f"   → {tokenizer.decode(output[0], skip_special_tokens=True)}")

# 2. Beam Search
print("\n2️⃣ Beam Search (num_beams=5):")
with torch.no_grad():
    output = model.generate(
        input_ids, 
        max_new_tokens=50, 
        num_beams=5,
        do_sample=False,
        no_repeat_ngram_size=2,  # 반복 방지
        **gen_kwargs,
    )
print(f"   → {tokenizer.decode(output[0], skip_special_tokens=True)}")

# 3. Random Sampling
print("\n3️⃣ Random Sampling (temperature=1.0):")
with torch.no_grad():
    output = model.generate(
        input_ids, 
        max_new_tokens=50, 
        do_sample=True,
        temperature=1.0,
        **gen_kwargs,
    )
print(f"   → {tokenizer.decode(output[0], skip_special_tokens=True)}")

# 4. Top-k Sampling
print("\n4️⃣ Top-k Sampling (k=50):")
with torch.no_grad():
    output = model.generate(
        input_ids, 
        max_new_tokens=50, 
        do_sample=True,
        top_k=50,
        **gen_kwargs,
    )
print(f"   → {tokenizer.decode(output[0], skip_special_tokens=True)}")

# 5. Top-p (Nucleus) Sampling
print("\n5️⃣ Top-p (Nucleus) Sampling (p=0.92):")
with torch.no_grad():
    output = model.generate(
        input_ids, 
        max_new_tokens=50, 
        do_sample=True,
        top_p=0.92,
        **gen_kwargs,
    )
print(f"   → {tokenizer.decode(output[0], skip_special_tokens=True)}")

print("\n" + "=" * 60)
print("✅ 같은 프롬프트에서도 전략에 따라 생성 결과가 달라집니다!")


In [ ]:
# Temperature 효과 비교
print("=" * 60)
print("🌡️ Temperature 효과 비교")
print("=" * 60)

prompt = "Once upon a time"
input_ids = tokenizer.encode(prompt, return_tensors="pt")
attention_mask = torch.ones_like(input_ids)

temperatures = [0.1, 0.5, 0.7, 1.0, 1.5]

for temp in temperatures:
    print(f"\n🌡️ Temperature = {temp}:")
    with torch.no_grad():
        output = model.generate(
            input_ids,
            attention_mask=attention_mask,
            pad_token_id=tokenizer.eos_token_id,
            max_new_tokens=40,
            do_sample=True,
            temperature=temp,
            top_p=0.95,
        )
    generated = tokenizer.decode(output[0], skip_special_tokens=True)
    print(f"   → {generated}")

print("\n" + "=" * 60)
print("💡 낮은 Temperature → 보수적/반복적")
print("💡 높은 Temperature → 창의적/불안정")
print("💡 일반적으로 0.7~0.9가 권장됩니다.")


---

## 5️⃣ 한국어 vs 영어 토큰 효율 비교

### 왜 한국어가 더 많은 토큰을 사용하는가?

- 🔸 대부분의 LLM은 **영어 중심** 학습 데이터로 토크나이저를 구축
- 🔸 영어 단어는 통째로 하나의 토큰이 되지만, 한글은 **자모 단위**로 분해
- 🔸 결과적으로 같은 의미의 한국어 문장이 **1.5~3배** 더 많은 토큰 사용
- 🔸 이는 **API 비용 증가**, **컨텍스트 윈도우 낭비**로 이어짐

### 한국어 최적화 모델의 등장

- Qwen2.5: 중국어+다국어 최적화 → 한국어도 상대적으로 효율적
- EXAONE: LG AI Research의 한국어 특화 모델
- SOLAR: Upstage의 한국어 최적화 모델

---

In [ ]:
# 한국어 vs 영어 토큰 효율 상세 비교
import tiktoken

print("=" * 60)
print("🇰🇷🇺🇸 한국어 vs 영어 토큰 효율 비교")
print("=" * 60)

enc = tiktoken.encoding_for_model("gpt-4")

# 동일한 의미의 문장 쌍
pairs = [
    ("Hello", "안녕하세요"),
    ("How are you?", "잘 지내시나요?"),
    ("I am a student.", "저는 학생입니다."),
    ("Artificial Intelligence", "인공지능"),
    ("Machine Learning", "머신러닝"),
    ("Natural Language Processing", "자연어 처리"),
    (
        "The transformer model uses self-attention mechanism to process sequences.",
        "트랜스포머 모델은 셀프 어텐션 메커니즘을 사용하여 시퀀스를 처리합니다."
    ),
    (
        "Fine-tuning allows us to adapt pre-trained models to specific tasks.",
        "파인튜닝을 통해 사전 학습된 모델을 특정 작업에 적응시킬 수 있습니다."
    ),
]

total_en = 0
total_ko = 0

print(f"\n{'영어':<50} {'토큰':>5} | {'한국어':<50} {'토큰':>5} | {'비율':>5}")
print("-" * 125)

for en, ko in pairs:
    en_tokens = len(enc.encode(en))
    ko_tokens = len(enc.encode(ko))
    ratio = ko_tokens / en_tokens if en_tokens > 0 else 0
    total_en += en_tokens
    total_ko += ko_tokens
    
    print(f"{en:<50} {en_tokens:>5} | {ko:<50} {ko_tokens:>5} | {ratio:>5.1f}x")

print("-" * 125)
print(f"{'합계':<50} {total_en:>5} | {'합계':<50} {total_ko:>5} | {total_ko/total_en:>5.1f}x")

print(f"\n📊 평균적으로 한국어는 영어의 약 {total_ko/total_en:.1f}배 토큰을 사용합니다.")
print(f"💡 API 비용으로 환산하면 한국어 처리가 약 {total_ko/total_en:.1f}배 더 비쌉니다!")

In [ ]:
# 한글 토큰화 상세 분석: 자모 분해 확인
print("=" * 60)
print("🔬 한글 토큰화 상세 분석")
print("=" * 60)

enc = tiktoken.encoding_for_model("gpt-4")

korean_words = ["안녕하세요", "인공지능", "파인튜닝", "트랜스포머", "대규모언어모델"]

for word in korean_words:
    tokens = enc.encode(word)
    decoded_parts = []
    for t in tokens:
        decoded = enc.decode([t])
        decoded_parts.append(repr(decoded))
    
    print(f"\n📝 '{word}'")
    print(f"   토큰 수: {len(tokens)}")
    print(f"   토큰 ID: {tokens}")
    print(f"   분해: {' | '.join(decoded_parts)}")

print("\n" + "=" * 60)
print("💡 한글은 UTF-8 바이트 단위로 쪼개져서 토큰이 많아집니다.")
print("💡 한국어 특화 모델(Qwen, EXAONE 등)은 이 문제를 개선했습니다.")

---

## 📝 정리 및 핵심 요약

### 이번 세션에서 배운 내용

| 주제 | 핵심 내용 |
|------|----------|
| 생성 AI | 새로운 콘텐츠를 만들어내는 AI, LLM은 텍스트 생성 |
| 트랜스포머 | Self-Attention 기반, Encoder/Decoder 구조 |
| 토큰화 | 텍스트→숫자 변환, BPE/WordPiece 등 다양한 방식 |
| 생성 전략 | Greedy, Beam Search, Top-k, Top-p, Temperature |
| 한국어 효율 | 영어 대비 1.5~3배 토큰 사용, 비용 및 컨텍스트 영향 |

### 다음 세션 예고

- 🔜 LLM 발전 과정과 주요 모델 비교
- 🔜 프롬프트 엔지니어링 / RAG / Agent / 파인튜닝 개요

---

### 💡 실습 과제

1. 자신의 이름과 자기소개를 한국어/영어로 작성하고, 토큰 수를 비교해보세요.
2. GPT-2 모델로 다양한 프롬프트에 대해 생성 전략별 결과를 비교해보세요.
3. (선택) 다른 토크나이저(예: `Qwen/Qwen2.5-1.5B-Instruct`)를 로딩하여 한국어 토큰 효율을 비교해보세요.

---